# Masked Face Verification: Pair-Head Final Notebook

This notebook packages the frozen main-branch project direction: a frozen FaceNet recognizer with a lightweight pair-level verifier head.

Final conclusion: the hypothesis is supported in a narrow but defensible form. The pair head improves masked-unmasked ROC-AUC ranking across held-out RMFD/RMFRD identity splits, preserves the unmasked path through a bypass, and remains below purpose-built mask-aware recognizers. Threshold calibration and larger-data exploration are future work on separate branches.

Default mode reads committed result artifacts. Set `RUN_EXPENSIVE = True` only to rerun the frozen probes on a GPU runtime.


## Setup

Use a GPU runtime. The scripts expect the RMFRD directory to contain masked and unmasked identity folders. The previously used Colab path was:

`/content/datasets/rmfrd/extracted/self-built-masked-face-recognition-dataset`

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/tungd/masked-face-id.git"
REPO_DIR = Path("/content/masked-face-id")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd /content/masked-face-id
!git pull --ff-only origin main
!python scripts/install_colab_deps.py

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/datasets/rmfrd/extracted/self-built-masked-face-recognition-dataset")
OUT_ROOT = Path("/content/masked_face_final_runs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Leave this False for a fast report/visualization notebook that uses committed artifacts.
# Set True only when the RMFD/RMFRD data is present and the runtime has a GPU.
RUN_EXPENSIVE = False

SEEDS = [42, 7, 99]
print({"data_root_exists": DATA_ROOT.exists(), "out_root": str(OUT_ROOT), "run_expensive": RUN_EXPENSIVE})


## Frozen Method

1. Align each face with MTCNN.
2. Extract frozen FaceNet embeddings for five views: full face, lower blackout, lower blur, upper-only, and eye-band.
3. Build pair features from view scores, score statistics, absolute embedding differences, and elementwise products.
4. Train a small MLP verifier head on training-identity pairs.
5. At inference, use the pair head for masked cases and bypass unmasked-unmasked pairs to the original FaceNet score.

This is the main-branch scope. Deeper FaceNet unfreezing, expanded synthetic benchmarks, and dataset upgrades should continue on separate branches.


In [ ]:
def run(cmd):
    print("$", " ".join(str(part) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)

if RUN_EXPENSIVE:
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")

    run([
        sys.executable, "scripts/probe_pair_head_robustness.py",
        "--data-root", DATA_ROOT,
        "--out-dir", OUT_ROOT / "pair_head_robustness_seed42_7_99",
        "--seeds", *SEEDS,
        "--train-identities", 300,
        "--eval-identities", 100,
        "--max-images-per-condition", 8,
        "--train-pairs-per-case", 10000,
        "--eval-pairs-per-case", 2000,
        "--epochs", 60,
    ])

    run([
        sys.executable, "-m", "pip", "install", "-q", "insightface", "onnxruntime-gpu",
    ])
    run([
        sys.executable, "scripts/probe_insightface_pair_head.py",
        "--data-root", DATA_ROOT,
        "--out-dir", OUT_ROOT / "insightface_pair_head_seed42",
        "--seeds", 42,
        "--train-identities", 300,
        "--eval-identities", 100,
        "--max-images-per-condition", 8,
        "--train-pairs-per-case", 10000,
        "--eval-pairs-per-case", 2000,
        "--epochs", 60,
    ])
else:
    print("Skipping expensive validation runs; using committed artifacts below.")


## Dedicated Mask-Aware Ceiling

The dedicated comparison uses official checkpoints from `fdbtrs/Masked-Face-Recognition-KD` with 112x112 ArcFace-style alignment. This is a ceiling model: it is allowed to be purpose-built for masked recognition.

The frozen main-branch claim does not depend on beating this ceiling. It uses the ceiling to define the remaining gap.


In [ ]:
CEILING_SEEDS = [42, 7]

if RUN_EXPENSIVE:
    for seed in CEILING_SEEDS:
        run([
            sys.executable, "scripts/probe_maskaware_baseline.py",
            "--data-root", DATA_ROOT,
            "--out-dir", OUT_ROOT / f"maskaware_seed{seed}",
            "--download-missing",
            "--seed", seed,
        ])
else:
    print("Skipping expensive dedicated-model runs; using committed historical ceiling artifacts below.")


In [ ]:
import pandas as pd
import numpy as np

ARTIFACTS = Path("artifacts")
ROBUST_DIR = ARTIFACTS / "rmfd_pair_head_robustness_seed42_7_99"
INSIGHT_DIR = ARTIFACTS / "insightface_pair_head_seed42"

robust = pd.read_csv(ROBUST_DIR / "pair_head_robustness_aggregate.csv")
threshold = pd.read_csv(ROBUST_DIR / "threshold_calibration_selected.csv")
insight = pd.read_csv(INSIGHT_DIR / "insightface_metrics.csv")

def row(model, case="masked-unmasked"):
    return robust[(robust["model"] == model) & (robust["case"] == case)].iloc[0]

baseline = row("baseline_full")
pair = row("pair_head_full_all_features_masked_only")
unmasked = row("pair_head_full_all_features_masked_only", "unmasked-unmasked")

summary = pd.DataFrame([
    {
        "model": "FaceNet baseline",
        "masked_unmasked_mean": baseline.roc_auc_mean,
        "std": baseline.roc_auc_std,
        "gain_vs_baseline": 0.0,
    },
    {
        "model": "Pair head masked-only",
        "masked_unmasked_mean": pair.roc_auc_mean,
        "std": pair.roc_auc_std,
        "gain_vs_baseline": pair.roc_auc_mean - baseline.roc_auc_mean,
    },
])

feature_ablation = robust[
    (robust["case"] == "masked-unmasked")
    & robust["model"].isin([
        "baseline_full",
        "pair_head_cosine_scores_only_masked_only",
        "pair_head_cosine_scores_stats_masked_only",
        "pair_head_dense_interactions_only_masked_only",
        "pair_head_full_face_dense_only_masked_only",
        "pair_head_full_all_features_masked_only",
    ])
][["model", "roc_auc_mean", "roc_auc_std"]].sort_values("roc_auc_mean", ascending=False)

threshold_summary = threshold.groupby("model")[["far", "tar", "accuracy"]].mean().reset_index()

print("Frozen hypothesis summary")
display(summary)
print("Feature ablation")
display(feature_ablation)
print("Calibration at nominal FAR 0.05")
display(threshold_summary)
print("InsightFace / ArcFace check")
display(insight)


In [ ]:
import matplotlib.pyplot as plt

plot_df = summary.copy()
fig, ax = plt.subplots(figsize=(8, 3.8))
colors = ["#596675", "#2364aa"]
labels = [
    f"FaceNet\n{baseline.roc_auc_mean:.4f} +/- {baseline.roc_auc_std:.4f}",
    f"Pair head\n{pair.roc_auc_mean:.4f} +/- {pair.roc_auc_std:.4f}",
]
ax.barh(labels, plot_df["masked_unmasked_mean"], xerr=plot_df["std"], color=colors)
ax.set_xlim(0.76, 0.85)
ax.set_xlabel("Masked-unmasked ROC-AUC")
ax.set_title("Frozen pair-head result: modest but consistent ranking gain")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
gain = pair.roc_auc_mean - baseline.roc_auc_mean
print(f"Masked-unmasked ROC-AUC: FaceNet {baseline.roc_auc_mean:.4f} +/- {baseline.roc_auc_std:.4f} -> pair head {pair.roc_auc_mean:.4f} +/- {pair.roc_auc_std:.4f} ({gain:+.4f}).")
print(f"Unmasked-unmasked is preserved by bypass at {unmasked.roc_auc_mean:.4f} +/- {unmasked.roc_auc_std:.4f}.")
print("Ablation conclusion: dense pair interactions drive the gain; cosine-only is essentially baseline.")
print("Calibration conclusion: better ROC-AUC does not yet translate to better TAR at fixed FAR 0.05.")
print("ArcFace check: direct-crop InsightFace buffalo_l is a negative control on this dataset, not a better replacement.")
print("\nFrozen framing: lightweight adaptation for existing FaceNet-style recognizers; future exploration belongs on separate branches.")


## Deliverables

- Report: `docs/final-report.md`
- Slides: `slides/pair_head_final_presentation.html`
- Frozen robustness probe: `scripts/probe_pair_head_robustness.py`
- InsightFace comparison probe: `scripts/probe_insightface_pair_head.py`
- Primary result artifacts: `artifacts/rmfd_pair_head_robustness_seed42_7_99/`
- InsightFace result artifacts: `artifacts/insightface_pair_head_seed42/`
- Historical ceiling artifacts: `artifacts/rmfrd_maskaware_baseline_*`

Main is frozen at this conclusion. Continue deeper FaceNet unfreezing, expanded synthetic benchmarks, and dataset upgrades on a new branch.
